# Gikomba Stall Analysis: Wanjiku's Fresh-Produce Business

**Analyst:** Hannah Wambui Muriu | **Cohort:** 1 Million Devs for Africa Software Engineering course - 1 Million Devs Cohort

Wanjiku runs a fresh-produce stall at Gikomba Market. She has two weeks of daily sales and a rough stock record, and wants to know: *How is my business doing? Is it growing? What should I restock, and where should I focus?*

This notebook uses **core Python only** (no pandas or NumPy), working with lists, dictionaries, slicing, functions and f-strings.

## Dataset
The data is copied exactly as provided. Everything below is calculated from these variables.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,

               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2



# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg

stock = {

    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},

    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},

    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},

    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},

    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},

}

## Task 1: Summary statistics for the 14 days
**Approach:** Use `sum()`, `len()`, `max()` and `min()` for the totals. To find *which* day was best or worst, use `list.index()` to get the position. The day name is `days[position]`, and the week number is `position // 7 + 1` (positions 0-6 are week 1, 7-13 are week 2).

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest = max(daily_sales)
lowest = min(daily_sales)

best_pos = daily_sales.index(highest)
worst_pos = daily_sales.index(lowest)
best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales    : KES {total_sales:,}")
print(f"Average / day  : KES {average_sales:,.2f}")
print(f"Highest day    : KES {highest:,} ({best_day}, week {best_week})")
print(f"Lowest day     : KES {lowest:,} ({worst_day}, week {worst_week})")

Total sales    : KES 46,450
Average / day  : KES 3,317.86
Highest day    : KES 5,600 (Sat, week 2)
Lowest day     : KES 1,950 (Tue, week 1)


## Task 2: Week totals and week-on-week growth
**Approach:** `week_totals()` slices the list into `[:7]` and `[7:14]`, sums each, and returns both as a tuple. The result is unpacked into two variables, and growth is `(week 2 - week 1) / week 1 * 100`.

In [3]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) for a list of 14 daily sales.

    Week 1 is the first 7 values (sales[:7]) and week 2 is the next 7 (sales[7:14]).
    """
    return sum(sales[:7]), sum(sales[7:14])


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total : KES {week1_total:,}")
print(f"Week 2 total : KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:+.2f}%")

Week 1 total : KES 22,050
Week 2 total : KES 24,400
Week-on-week growth: +10.66%


## Task 3: Classify each day as Strong, Normal or Weak
**Approach:** `classify_day()` compares a day's sales to the average using 20% thresholds (above 1.2 x average is *Strong*, below 0.8 x average is *Weak*, anything else is *Normal*). I apply it to every day and count the categories with the dictionary counting pattern (`counts[label] = counts.get(label, 0) + 1`).

In [4]:
def classify_day(amount, average):
    """Classify a day's sales relative to the average.

    Returns "Strong" if amount is more than 20% above average,
    "Weak" if it is more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    return "Normal"


day_labels = [classify_day(amount, average_sales) for amount in daily_sales]

category_counts = {}
for label in day_labels:
    category_counts[label] = category_counts.get(label, 0) + 1

print(f"Strong threshold: above KES {average_sales * 1.2:,.2f}")
print(f"Weak threshold  : below KES {average_sales * 0.8:,.2f}\n")
for i, (day, amount, label) in enumerate(zip(days, daily_sales, day_labels)):
    print(f"Week {i // 7 + 1} {day}: KES {amount:>6,}  -> {label}")
print("\nCounts:", category_counts)

Strong threshold: above KES 3,981.43
Weak threshold  : below KES 2,654.29

Week 1 Mon: KES  2,300  -> Weak
Week 1 Tue: KES  1,950  -> Weak
Week 1 Wed: KES  2,800  -> Normal
Week 1 Thu: KES  2,100  -> Weak
Week 1 Fri: KES  3,600  -> Normal
Week 1 Sat: KES  5,200  -> Strong
Week 1 Sun: KES  4,100  -> Strong
Week 2 Mon: KES  2,500  -> Weak
Week 2 Tue: KES  2,200  -> Weak
Week 2 Wed: KES  3,100  -> Normal
Week 2 Thu: KES  2,400  -> Weak
Week 2 Fri: KES  3,900  -> Normal
Week 2 Sat: KES  5,600  -> Strong
Week 2 Sun: KES  4,700  -> Strong

Counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays vs weekends
**Approach:** Because `days` repeats Mon-Sun, a day is a weekend if its name is `"Sat"` or `"Sun"`. I split the sales into two lists in one loop, average each, and express the weekend premium as a percentage of the weekday average.

In [5]:
weekday_sales = []
weekend_sales = []
for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_premium_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}  ({len(weekday_sales)} days)")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}  ({len(weekend_sales)} days)")
print(f"Weekends earn {weekend_premium_pct:.1f}% more per day than weekdays")

Weekday average (Mon-Fri): KES 2,685.00  (10 days)
Weekend average (Sat-Sun): KES 4,900.00  (4 days)
Weekends earn 82.5% more per day than weekdays


## Task 5: Stock reorder list and gross margins
**Approach:** `products_to_reorder()` loops over `stock.items()` and collects the product names where `qty <= reorder`. Gross margin % is `(price - cost) / price * 100`, calculated for each product in a second loop and stored in a dictionary.

In [6]:
def products_to_reorder(stock_data):
    """Return a list of product names whose quantity is at or below the reorder level."""
    return [name for name, info in stock_data.items() if info["qty"] <= info["reorder"]]


reorder_list = products_to_reorder(stock)

margins = {}
for name, info in stock.items():
    margins[name] = (info["price"] - info["cost"]) / info["price"] * 100

print("Products to reorder:", reorder_list)
for name in reorder_list:
    info = stock[name]
    print(f"  {name}: {info['qty']} kg in stock, reorder level {info['reorder']} kg "
          f"(short by {info['reorder'] - info['qty']} kg)")

print("\nGross margin by product:")
for name, margin in margins.items():
    print(f"  {name:<10} {margin:5.1f}%")

Products to reorder: ['Tomatoes', 'Sukuma']
  Tomatoes: 18 kg in stock, reorder level 25 kg (short by 7 kg)
  Sukuma: 9 kg in stock, reorder level 15 kg (short by 6 kg)

Gross margin by product:
  Tomatoes    33.3%
  Onions      30.0%
  Sukuma      40.0%
  Potatoes    33.3%
  Avocado     42.9%


## Task 6: Stall Performance Report
**Approach:** Combine all results into one report using f-strings. Format specifiers handle alignment (`:<`, `:>`) and thousands separators (`:,`), and money is always shown as `KES`.

In [7]:
width = 52
best_margin_name = max(margins, key=margins.get)

print("=" * width)
print("STALL PERFORMANCE REPORT".center(width))
print("Wanjiku's Fresh Produce - Gikomba Market".center(width))
print("=" * width)

print("\nSALES SUMMARY (14 days)")
print("-" * width)
print(f"{'Total sales':<26}{'KES ' + format(total_sales, ','):>26}")
print(f"{'Average per day':<26}{'KES ' + format(average_sales, ',.2f'):>26}")
print(f"{'Best day':<26}{f'KES {highest:,} ({best_day}, Wk {best_week})':>26}")
print(f"{'Worst day':<26}{f'KES {lowest:,} ({worst_day}, Wk {worst_week})':>26}")

print("\nGROWTH")
print("-" * width)
print(f"{'Week 1 total':<26}{'KES ' + format(week1_total, ','):>26}")
print(f"{'Week 2 total':<26}{'KES ' + format(week2_total, ','):>26}")
print(f"{'Week-on-week growth':<26}{format(growth_pct, '+.2f') + '%':>26}")

print("\nDAY PERFORMANCE")
print("-" * width)
for label in ("Strong", "Normal", "Weak"):
    print(f"{label + ' days':<26}{category_counts.get(label, 0):>26}")

print("\nWEEKDAY vs WEEKEND")
print("-" * width)
print(f"{'Weekday avg (Mon-Fri)':<26}{'KES ' + format(weekday_avg, ',.2f'):>26}")
print(f"{'Weekend avg (Sat-Sun)':<26}{'KES ' + format(weekend_avg, ',.2f'):>26}")
print(f"{'Weekend premium':<26}{format(weekend_premium_pct, '.1f') + '%':>26}")

print("\nSTOCK & MARGINS")
print("-" * width)
print(f"{'Product':<12}{'Stock':>8}{'Reorder':>9}{'Margin':>9}{'Status':>14}")
for name, info in stock.items():
    status = "REORDER" if name in reorder_list else "OK"
    print(f"{name:<12}{str(info['qty']) + ' kg':>8}{str(info['reorder']) + ' kg':>9}"
          f"{margins[name]:>8.1f}%{status:>14}")
print("-" * width)
print(f"Reorder now: {', '.join(reorder_list)}")
print(f"Best margin: {best_margin_name} ({margins[best_margin_name]:.1f}%)")
print("=" * width)

              STALL PERFORMANCE REPORT              
      Wanjiku's Fresh Produce - Gikomba Market      

SALES SUMMARY (14 days)
----------------------------------------------------
Total sales                               KES 46,450
Average per day                         KES 3,317.86
Best day                       KES 5,600 (Sat, Wk 2)
Worst day                      KES 1,950 (Tue, Wk 1)

GROWTH
----------------------------------------------------
Week 1 total                              KES 22,050
Week 2 total                              KES 24,400
Week-on-week growth                          +10.66%

DAY PERFORMANCE
----------------------------------------------------
Strong days                                        4
Normal days                                        4
Weak days                                          6

WEEKDAY vs WEEKEND
----------------------------------------------------
Weekday avg (Mon-Fri)                   KES 2,685.00
Weekend avg (Sat-Sun)        

## Recommendations for Wanjiku

1. **Restock Tomatoes and Sukuma today.** Both are below their reorder levels: Tomatoes has 18 kg against a 25 kg level (short by 7 kg) and Sukuma has 9 kg against 15 kg (short by 6 kg). Sukuma also earns a 40.0% margin, so running out costs her good money.
2. **Build stock for the weekend.** Weekends average KES 4,900 per day versus KES 2,685 on weekdays, which is 82.5% more, and all 4 "Strong" days were Saturdays or Sundays. Buy extra stock on Thursday or Friday so she never sells out on her best days.
3. **Push Avocado, her highest-margin product.** Avocado earns a 42.9% margin (KES 60 profit per kg) compared with 30.0% for Onions, but only 12 kg is in stock, just above its 10 kg reorder level. Keep it well stocked and display it prominently.
4. **Fix the slow days with promotions.** 6 of the 14 days were "Weak" (below KES 2,654), all on a Monday, Tuesday or Thursday, with Tuesday the lowest at KES 1,950. A small midweek offer, such as a bundle of onions and tomatoes, could lift these days.
5. **The business is growing, so keep going.** Week 2 sales were KES 24,400 against KES 22,050 in week 1, which is growth of 10.66%. Stock levels should be sized for the higher week-2 level, not the week-1 level.